In [3]:

import os
import json
import re
from dotenv import load_dotenv
from openai import OpenAI

# Bring the Anthropic class from the anthropic package — I want to connect to Claude AI.

# from anthropic import Anthropic
from IPython.display import Markdown, display

# Always remember to do this!
load_dotenv(override=True)
google_api_key = os.getenv('GOOGLE_API_KEY')
if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")
request = "Please come up with a challenging, nuanced question on topic of AI/ML that I can ask a number of LLMs to evaluate their intelligence. "
request += "Answer only with the question, no explanation."
messages = [{"role": "user", "content": request}]
gemini = OpenAI(api_key=google_api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
response = gemini.chat.completions.create(
    model="gemini-2.5-flash",
    messages=messages,
)
question = response.choices[0].message.content
display(Markdown(question))
competitors = []
answers = []
messages = [{"role": "user", "content": question}]
# The API we know well
# I've updated this with the latest model, but it can take some time because it likes to think!
# Replace the model with gpt-4.1-mini if you'd prefer not to wait 1-2 mins

model_name = "gemini-2.5-flash"

response = gemini.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

# competitors = ["Google", "Amazon", "Microsoft"]

# answers = ["Search engine", "E-commerce", "Software"]

# for c, a in zip(competitors, answers):

#     print(c, a)

# Output:

# Google Search engine

# Amazon E-commerce

# Microsoft Software

for competitor, answer in zip(competitors, answers):
    print(f"Competitor: {competitor}\n\n{answer}")

# answers = ["A", "B", "C"]

# for index, value in enumerate(answers):

#     print(index, value)

# Output:

# 0 A,
# 1 B,
# 2 C
# Let's bring this together - note the use of "enumerate"

together = ""
for index, answer in enumerate(answers):
    together += f"# Response from competitor {index+1}\n\n"
    together += answer + "\n\n"

    judge = f"""You are judging a competition between {len(competitors)} competitors.
Each model has been given this question:

{question}

Your job is to evaluate each response for clarity and strength of argument, and rank them in order of best to worst.
Respond with JSON, and only JSON, with the following format:
{{"results": ["best competitor number", "second best competitor number", "third best competitor number", ...]}}

Here are the responses from each competitor:

{together}

Now respond with the JSON with the ranked order of the competitors, nothing else. Do not include markdown formatting or code blocks."""
judge_messages = [{"role": "user", "content": judge}]
openai = OpenAI(api_key=google_api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
response = openai.chat.completions.create(
    model="gemini-2.5-flash",
    messages=judge_messages,
)
results = response.choices[0].message.content
print(results)

# OK let's turn this into results!

results_dict = json.loads(results)
ranks = results_dict["results"]
for index, result in enumerate(ranks):
    # Extract number from "competitor 1", "1", etc. (LLM may return different formats)
    match = re.search(r'\d+', str(result))
    competitor_num = int(match.group()) if match else int(result)
    competitor = competitors[competitor_num - 1]
    print(f"Rank {index+1}: {competitor}")

Google API Key exists and begins AI


Given the increasing capacity of AI models to generate novel outputs—ranging from scientific hypotheses to artistic works—that often lack straightforward attribution to either their training data or human programmers, how does this phenomenon compel a re-evaluation of our established frameworks for intellectual property, ethical accountability, and the fundamental definition of human ingenuity in the era of advanced artificial intelligence?

The increasing capacity of AI models to generate novel outputs profoundly challenges our established frameworks across intellectual property, ethical accountability, and the very definition of human ingenuity. This phenomenon compels a re-evaluation because the traditional assumptions underpinning these frameworks—namely, a clear human author, agent, or creator—are fundamentally disrupted when the origin of a novel output is complex, opaque, or non-human.

Here's a breakdown of how this phenomenon compels a re-evaluation:

### I. Intellectual Property (IP)

**Established Frameworks:** IP law (copyright, patent, trade secret) is predicated on the idea of a human creator or inventor. It grants exclusive rights to encourage innovation and reward original authorship. Concepts like "originality," "inventiveness," and "authorship" are tied to human intent, consciousness, and effort.

**Challenges Posed by AI-Generated Outputs:**

1.  **Authorship and Ownership:**
    *   **Who is the "author"?** Is it the AI itself (which lacks legal personhood)? The developer who created the AI? The user who prompted it? The entity that trained it on vast datasets? Or is it none of the above?
    *   **Lack of "Human Originality":** Copyright requires "original works of authorship," typically interpreted as creations flowing from a human intellect. AI operates through algorithms and data patterns, not consciousness or intent. Can an AI truly be original in the human sense?
    *   **Derivative Works vs. Fair Use:** If an AI is trained on copyrighted material, and its output resembles or is inspired by that material, is it an infringing derivative work, or a transformative new creation akin to fair use? The scale and complexity of AI training make traditional analysis extremely difficult.

**Compelled Re-evaluation:**

*   **New Categories of IP:** We may need entirely new categories of IP protection, perhaps "AI-assisted IP" or "Algorithmic IP," with different durations and rights.
*   **Layered Ownership Models:** A system that attributes ownership across multiple stakeholders (developer, trainer, user, data providers) or defaults certain AI-generated works to the public domain.
*   **"Legal Personhood" for AI?** While controversial, some legal scholars explore the idea of granting AI limited legal personhood for IP purposes, or creating a new legal entity class.
*   **Transparency and Attribution:** Mandating disclosure of AI involvement in creation, and potentially tracing back to training data sources for licensing or attribution.
*   **Redefining "Originality":** Shifting the legal definition of originality to include outputs generated by sophisticated algorithms, perhaps focusing on the "novelty" or "usefulness" of the output itself, rather than the nature of its creator.

### II. Ethical Accountability

**Established Frameworks:** Ethical and legal accountability frameworks are designed to attribute responsibility and liability to human agents or organizations for their actions and their consequences. Concepts like intent, negligence, foresight, and culpability are central to determining blame and imposing sanctions.

**Challenges Posed by AI-Generated Outputs:**

1.  **Attribution Gap:** When an AI generates harmful content (e.g., deepfakes, misinformation, biased scientific "hypotheses," dangerous instructions) or makes decisions leading to adverse outcomes, attributing blame becomes incredibly complex.
    *   **"Black Box" Problem:** Advanced AI models can produce outputs through non-transparent processes, making it difficult to understand *why* a particular output was generated or a decision was made.
    *   **Unintended Consequences:** AI can exhibit emergent behaviors not explicitly programmed or foreseen by its developers, leading to outputs that no human "intended."
    *   **Diffusion of Responsibility:** Is it the AI designer, the data annotator, the AI trainer, the deployer, the user, or even the AI itself that is ethically or legally responsible?

**Compelled Re-evaluation:**

*   **Layered Liability Models:** Developing legal frameworks that distribute responsibility across the AI's lifecycle, from design and development to deployment and usage. This could involve strict liability for developers/deployers for certain types of harm.
*   **"Ethical AI by Design":** Mandating that ethical considerations (fairness, transparency, safety) are built into AI systems from the outset, with accountability mechanisms integrated.
*   **Independent Auditing and Oversight:** Establishing regulatory bodies or independent auditors to review AI systems for bias, safety, and adherence to ethical guidelines, especially for critical applications.
*   **Explainable AI (XAI):** Research and development into making AI decision-making processes more transparent and interpretable, reducing the "black box" problem to facilitate accountability.
*   **Defining "Reasonable Foreseeability" for AI:** How do we adapt legal concepts of negligence and foreseeability when AI can generate outputs or take actions that are difficult for humans to predict?

### III. The Fundamental Definition of Human Ingenuity

**Established Frameworks:** Human ingenuity has traditionally been defined by our unique capacity for creativity, innovation, problem-solving, insight, artistic expression, and abstract thought. It's often linked to consciousness, subjective experience, and the spark of genuine inspiration. Our sense of self-worth and societal progress are deeply intertwined with this understanding.

**Challenges Posed by AI-Generated Outputs:**

1.  **Mimicry of Human Creativity:** AI can now generate music, art, poetry, scientific hypotheses, and architectural designs that are virtually indistinguishable from (or even surpass) human creations in terms of aesthetic appeal or functional utility. This blurs the line between human and machine creativity.
2.  **Redefining "Creativity":** If AI can produce novel, valuable outputs without consciousness, intent, or the human experience of struggle and inspiration, does this change what "creativity" truly means? Is it merely the production of novel combinations, or does it require something more profound?
3.  **Existential Questions:** If machines can outperform humans in tasks once considered uniquely human (e.g., grandmaster chess, complex scientific discovery, artistic composition), what then is the unique value proposition of human intellect and spirit? Does it devalue human achievement?
4.  **The "Tool vs. Creator" Debate:** Is AI merely a sophisticated tool that augments human ingenuity, or is it a new form of intelligent agent capable of genuine independent ingenuity?

**Compelled Re-evaluation:**

*   **Shifting Focus of Human Ingenuity:** Rather than focusing solely on the *output*, we might redefine human ingenuity by emphasizing the *process* of creation—the intent, the emotional depth, the unique subjective experience, the critical framing of problems, and the ability to imbue work with meaning and empathy.
*   **AI as a Collaborator/Augmentation:** Viewing AI not as a replacement, but as a powerful new collaborator that extends human capabilities, allowing us to explore new frontiers of creativity and discovery at unprecedented scales and speeds.
*   **The Uniquely Human Role:** Emphasizing human roles in setting ethical parameters for AI, defining overarching goals, interpreting AI-generated insights, providing critical judgment, and integrating AI outputs into a broader human context of meaning and purpose.
*   **Reclaiming "Consciousness" and "Intent":** Reasserting that true ingenuity, at its deepest level, still requires consciousness, self-awareness, and intentionality—qualities that current AI models do not possess.
*   **Philosophical Re-assessment:** Engaging in a deeper philosophical inquiry into the nature of intelligence, creativity, and consciousness in a world where artificial systems can mimic so much of what we once thought exclusive to humanity.

In conclusion, the advent of sophisticated AI compels not just minor adjustments but a fundamental re-evaluation of the bedrock principles underlying our societal and legal structures. We are forced to confront the limitations of frameworks built for a pre-AI world and to grapple with profound questions about authorship, responsibility, and the very essence of human distinctiveness in the age of intelligent machines. This re-evaluation demands interdisciplinary collaboration among technologists, legal scholars, ethicists, philosophers, and policymakers to design adaptive, resilient, and equitable frameworks for the future.

Competitor: gemini-2.5-flash

The increasing capacity of AI models to generate novel outputs profoundly challenges our established frameworks across intellectual property, ethical accountability, and the very definition of human ingenuity. This phenomenon compels a re-evaluation because the traditional assumptions underpinning these frameworks—namely, a clear human author, agent, or creator—are fundamentally disrupted when the origin of a novel output is complex, opaque, or non-human.

Here's a breakdown of how this phenomenon compels a re-evaluation:

### I. Intellectual Property (IP)

**Established Frameworks:** IP law (copyright, patent, trade secret) is predicated on the idea of a human creator or inventor. It grants exclusive rights to encourage innovation and reward original authorship. Concepts like "originality," "inventiveness," and "authorship" are tied to human intent, consciousness, and effort.

**Challenges Posed by AI-Generated Outputs:**

1.  **Authorship and Ownership:*

ValueError: invalid literal for int() with base 10: 'competitor 1'